# NOT training: Handbags → Shoes

Ported from the paper experiment (`handbags_to_shoes/` + `notebooks/NOT_training_strong.py`): source = pix2pix `edges2handbags` photos rendered red, target = pix2pix `edges2shoes` photos rendered blue, trained with the repo's generic `train_gdmax` / `train_extragradient`.

**Data not included.** Download it first:
```bash
bash ../scripts/download_product_data.sh
```
This populates `../data/edges2handbags/` and `../data/edges2shoes/`. Until then, the data-loading cell below will raise `RuntimeError: No product images found`.

In [ ]:
import os, sys
sys.path.append("..")

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from IPython.display import clear_output

device = "cuda" if torch.cuda.is_available() else "cpu"

from src.models import TransportUNet, PotentialResNet, warmup_spectral_norm
from src.datasets import build_handbags_to_shoes, LoaderSampler
from src.utils import set_seed, warmup_cosine_decay, plot_loss, plot_grad_norm, show_image_samples
from src.train import train_gdmax, train_extragradient

SEED = 0
IMG_SIZE = 64
BATCH_SIZE = 64

# Warm-up through WARMUP_STEPS, max LR through DECAY_START, cosine decay to
# MIN_LR_FRAC * max through DECAY_END, floor after -- matches the paper schedule.
WARMUP_STEPS = 1_000
DECAY_START = 20_000
DECAY_END = 150_000
MIN_LR_FRAC = 0.05
N_STEPS = 160_000

EXP_DIR = "../outputs/handbags_to_shoes"
GDMAX_DIR = os.path.join(EXP_DIR, "gdmax")
EG_DIR = os.path.join(EXP_DIR, "eg")
os.makedirs(GDMAX_DIR, exist_ok=True)
os.makedirs(EG_DIR, exist_ok=True)


## Data: red handbags → blue shoes

The RGB-photo half of each pix2pix edge|photo pair is cropped, resized to 64×64, and projected onto a single color channel: source handbags keep only R (G, B forced to +1, the white background value), target shoes keep only B.

In [ ]:
source_data, target_data = build_handbags_to_shoes("../data", img_size=IMG_SIZE)
print(f"handbag photos: {len(source_data):,}")
print(f"shoe photos: {len(target_data):,}")

source_sampler = LoaderSampler(source_data, batch_size=BATCH_SIZE, seed=SEED, device=device)
target_sampler = LoaderSampler(target_data, batch_size=BATCH_SIZE, seed=SEED + 1, device=device)


def sample_mu(batch_size):
    return source_sampler.sample()


def sample_nu(batch_size):
    return target_sampler.sample()


# Fixed images so every saved figure compares the same source/target samples.
fixed_x_sampler = LoaderSampler(source_data, batch_size=10, seed=SEED + 10_000, device=device)
fixed_y_sampler = LoaderSampler(target_data, batch_size=10, seed=SEED + 10_001, device=device)
X_FIXED = fixed_x_sampler.sample()
Y_FIXED = fixed_y_sampler.sample()

cost = F.mse_loss
ADAM_KWARGS = dict(betas=(0.0, 0.9), weight_decay=1e-10, amsgrad=True)
GRAD_CLIP = 10.0


def make_lr_schedule(lr_T_max, lr_f_max):
    def lr_schedule(step):
        return (
            warmup_cosine_decay(step, WARMUP_STEPS, DECAY_START, DECAY_END, lr_T_max, MIN_LR_FRAC),
            warmup_cosine_decay(step, WARMUP_STEPS, DECAY_START, DECAY_END, lr_f_max, MIN_LR_FRAC),
        )
    return lr_schedule


## GDmax Training

In [ ]:
set_seed(SEED)
T_gdmax = TransportUNet(base_channels=64).to(device)
f_gdmax = PotentialResNet(base_channels=64).to(device)
warmup_spectral_norm(f_gdmax, IMG_SIZE, device=device)

history_gdmax = train_gdmax(
    T_gdmax, f_gdmax, sample_mu, sample_nu, cost,
    n_steps=N_STEPS, lr_T=5e-5, lr_f=5e-5, K=5,
    batch_size_x=BATCH_SIZE, batch_size_y=BATCH_SIZE,
    lr_schedule=make_lr_schedule(5e-5, 5e-5),
    grad_clip=GRAD_CLIP, optimizer_kwargs=ADAM_KWARGS,
    log_dir=GDMAX_DIR,
    callback=lambda step, T, f: show_image_samples(
        T, X_FIXED, Y_FIXED, method_name="gdmax",
        save_path=os.path.join(GDMAX_DIR, f"samples_step_{step + 1:06d}.png"),
    ),
)


In [ ]:
plot_loss(history_gdmax["loss"], save_path=os.path.join(GDMAX_DIR, "loss.png"))
plot_grad_norm(history_gdmax["grad_T"], history_gdmax["grad_f"], save_path=os.path.join(GDMAX_DIR, "grad_norm.png"))
show_image_samples(T_gdmax, X_FIXED, Y_FIXED, method_name="gdmax", save_path=os.path.join(GDMAX_DIR, "final_samples.png"))


## Extragradient Training

In [ ]:
set_seed(SEED)
T_eg = TransportUNet(base_channels=64).to(device)
f_eg = PotentialResNet(base_channels=64).to(device)
warmup_spectral_norm(f_eg, IMG_SIZE, device=device)

history_eg = train_extragradient(
    T_eg, f_eg, sample_mu, sample_nu, cost,
    n_steps=N_STEPS, lr_T=1e-4, lr_f=1e-4,
    batch_size_x=BATCH_SIZE, batch_size_y=BATCH_SIZE,
    lr_schedule=make_lr_schedule(1e-4, 1e-4),
    grad_clip=GRAD_CLIP, optimizer_kwargs=ADAM_KWARGS,
    log_dir=EG_DIR,
    callback=lambda step, T, f: show_image_samples(
        T, X_FIXED, Y_FIXED, method_name="eg",
        save_path=os.path.join(EG_DIR, f"samples_step_{step + 1:06d}.png"),
    ),
)


In [ ]:
plot_loss(history_eg["loss"], save_path=os.path.join(EG_DIR, "loss.png"))
plot_grad_norm(history_eg["grad_T"], history_eg["grad_f"], save_path=os.path.join(EG_DIR, "grad_norm.png"))
show_image_samples(T_eg, X_FIXED, Y_FIXED, method_name="eg", save_path=os.path.join(EG_DIR, "final_samples.png"))
